# Workshop 3.2: Grouping on MultiIndex DataFrames

Welcome to Workshop 3.2! In Workshop 3.1, we discovered how to stack multiple assets into a single unified MultiIndex DataFrame. While a MultiIndex keeps our portfolio tidy, performing time-series calculations across multiple stocks requires strict care.

### The Silent Threat of Cross-Asset Leakage

In a single-stock DataFrame, computing returns with `.pct_change()` is safe because every row belongs to the same asset. But in a multi-asset table, adjacent rows on the same date belong to completely different companies.

If you naively call `.pct_change()` on an unpartitioned MultiIndex, Python divides Apple's closing price by Amazon's previous price. This catastrophic mistake is known as **cross-asset leakage**. It generates fictional returns without raising an error, quietly corrupting downstream backtests.

To calculate features safely, we must isolate each asset's time series using `.groupby(level="Ticker")`.

In this workshop, we will examine cross-asset leakage, master grouped feature engineering with `.transform()`, and learn how to pivot tables using `.unstack()`.

> **Key Takeaway**: Always calculate time-series features within `.groupby(level="Ticker")` to prevent cross-asset leakage between different stocks.

## Topic 1: The Danger of Cross-Asset Leakage

To understand why grouping is mandatory, let's observe what happens when we calculate percentage changes directly on an unpartitioned MultiIndex DataFrame.

Let's load a two-stock universe of Apple and Microsoft. Let's see:

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Re-create build_universe helper from Workshop 3.1:
def build_universe(ticker_list, start_date, end_date):
    frames = []
    for ticker in ticker_list:
        df = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df["Ticker"] = ticker
        frames.append(df)
    combined = pd.concat(frames)
    combined = combined.reset_index().set_index(["Date", "Ticker"]).sort_index()
    return combined

# Build a small 2-stock universe:
small_universe = build_universe(["AAPL", "MSFT"], "2023-01-01", "2023-01-10")

# WRONG: Computing returns without grouping by ticker:
small_universe["Wrong_Return"] = small_universe["Close"].pct_change()

print("Flawed return calculation without GroupBy:")
print(small_universe[["Close", "Wrong_Return"]].head(6))

Demonstrating Cross-Asset Leakage (The Wrong Way):
Price                   Close  Wrong_Return
Date       Ticker                          
2023-01-03 AAPL    122.876755           NaN
           MSFT    232.510544      0.892226
2023-01-04 AAPL    124.144127     -0.466071
           MSFT    222.339783      0.790981
2023-01-05 AAPL    122.827606     -0.447568
           MSFT    215.750168      0.756528
2023-01-06 AAPL    127.346970     -0.409748
           MSFT    218.292816      0.714158
2023-01-09 AAPL    127.867676     -0.414238
           MSFT    220.418213      0.723799


### Deconstructing the Contamination

Look closely at row 1 on `2023-01-03`. The first row is Apple at `$125.07`, while the second row is Microsoft at `$239.58`. When pandas calculated `Wrong_Return` on row 1, it divided Microsoft's price by Apple's price, reporting a fictional `+91.56%` return.

This illustrates why unpartitioned multi-asset calculations are toxic: the code runs without complaint, but the math is completely invalid.

> **Key Takeaway**: Calling `.pct_change()` without grouping compares adjacent rows across different assets, corrupting returns entirely.

## Topic 2: The Safe Approach: .groupby(level="Ticker")

To compute features safely, we instruct pandas to isolate each asset's time series using **`.groupby(level="Ticker")`**.

Grouping by the `Ticker` index level creates parallel processing lanes: all Apple rows are evaluated strictly against prior Apple rows, and all Microsoft rows remain in their own independent sequence.

Let's calculate daily returns using proper ticker partitioning. Let's check:

In [2]:
# RIGHT: Group by the Ticker level before computing returns:
small_universe["Correct_Return"] = small_universe.groupby(level="Ticker")["Close"].pct_change()

print("Correct return calculation using .groupby(level='Ticker'):")
print(small_universe[["Close", "Wrong_Return", "Correct_Return"]].head(6))

Comparing Wrong_Return vs Correct Return:
Price                   Close  Wrong_Return    Return
Date       Ticker                                    
2023-01-03 AAPL    122.876755           NaN       NaN
           MSFT    232.510544      0.892226       NaN
2023-01-04 AAPL    124.144127     -0.466071  0.010314
           MSFT    222.339783      0.790981 -0.043743
2023-01-05 AAPL    122.827606     -0.447568 -0.010605
           MSFT    215.750168      0.756528 -0.029638
2023-01-06 AAPL    127.346970     -0.409748  0.036794
           MSFT    218.292816      0.714158  0.011785
2023-01-09 AAPL    127.867676     -0.414238  0.004089
           MSFT    220.418213      0.723799  0.009736


> **Key Takeaway**: Grouping by `level="Ticker"` ensures calculations remain strictly confined to each asset's historical timeline.

---

## Topic 3: Adding Features with .transform()

When building quantitative signals, we frequently compute multi-day features like moving averages or rolling volatility.

For complex window calculations within a grouped MultiIndex, pandas provides **`.transform()`**:
```python
grouped["Close"].transform(lambda x: x.rolling(20).mean())
```

The `.transform()` method applies our custom calculation to each ticker group separately, returning a Series that matches the original MultiIndex structure perfectly.

Let's compute 5-day returns and 20-day moving averages across our five-asset universe. Let's see:

In [3]:
# Build full 5-stock universe for 2023:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
universe = build_universe(tickers, "2023-01-01", "2024-01-01")

# Group by Ticker level:
grouped = universe.groupby(level="Ticker")

# Add 5-day return and 20-day SMA using GroupBy:
universe["Return_5d"] = grouped["Close"].pct_change(5)
universe["SMA_20"] = grouped["Close"].transform(lambda x: x.rolling(20).mean())

# Show the first few rows:
print("First few rows showing Return_5d and SMA_20:")
print(universe[["Close", "Return_5d", "SMA_20"]].dropna().head(10))

First few rows showing Return_5d and SMA_20:
Price                   Close  Return_5d      SMA_20
Date       Ticker                                   
2023-01-31 AAPL    141.759674   0.012348  133.397947
           AMZN    103.129997   0.070702   94.223500
           GOOGL    97.971268   0.011668   92.222741
           MSFT    240.497711   0.023839  230.279390
           NVDA     19.493687   0.014119   17.235005
2023-02-01 AAPL    142.879700   0.015242  134.398095
           AMZN    105.150002   0.081827   95.190000
           GOOGL    99.547295   0.046039   92.783270
           MSFT    245.291931   0.038167  230.918459
           NVDA     20.896568   0.147690   17.565670


> **Key Takeaway**: `.transform()` evaluates rolling calculations separately per ticker while preserving the original MultiIndex shape.

---

## Topic 4: Lagging Signals Across MultiIndex Universes

Just as with single-stock models, we must lag trading signals by one day to prevent look-ahead bias.

On a MultiIndex table, we **must shift within each ticker group**. Calling `.shift(1)` without grouping causes signals from one stock to bleed into the following stock's trading position.

Let's generate trading signals and lag them within each ticker partition. Let's see:

In [4]:
# Generate raw signal (1 if Close > SMA_20, else 0):
universe["Signal"] = (universe["Close"] > universe["SMA_20"]).astype(int)

# Lag the signal by 1 day within each ticker group:
universe["Position"] = universe.groupby(level="Ticker")["Signal"].shift(1)

# Print a comparison of Signal vs Position:
print("Comparison of Signal vs Position across tickers:")
print(universe[["Close", "SMA_20", "Signal", "Position"]].dropna().head(10))

Comparison of Signal vs Position across tickers:
Price                   Close      SMA_20  Signal  Position
Date       Ticker                                          
2023-01-31 AAPL    141.759674  133.397947       1       0.0
           AMZN    103.129997   94.223500       1       0.0
           GOOGL    97.971268   92.222741       1       0.0
           MSFT    240.497711  230.279390       1       0.0
           NVDA     19.493687   17.235005       1       0.0
2023-02-01 AAPL    142.879700  134.398095       1       1.0
           AMZN    105.150002   95.190000       1       1.0
           GOOGL    99.547295   92.783270       1       1.0
           MSFT    245.291931  230.918459       1       1.0
           NVDA     20.896568   17.565670       1       1.0


> **Key Takeaway**: Always shift trading signals inside `.groupby(level="Ticker")` to keep position lags isolated to their respective assets.

---

## Topic 5: Cross-Sectional Pivoting with .unstack()

Sometimes our research requires comparing all assets simultaneously on the same date, such as evaluating relative momentum or calculating correlation matrices.

We can pivot our long MultiIndex table into a wide format using **`.unstack(level="Ticker")`**.

Think of `.unstack()` like rotating a table so that each unique stock ticker becomes its own column header across dates.

Let's pivot our daily returns and compute the asset correlation matrix. Let's check:

In [5]:
# Calculate daily returns grouped by ticker:
universe["Return"] = universe.groupby(level="Ticker")["Close"].pct_change()

# Code: Pivot Ticker level to columns:
returns_wide = universe["Return"].unstack(level="Ticker")

# Print the wide DataFrame:
print("Wide Returns DataFrame (first 5 rows):")
print(returns_wide.head())

# Calculate correlations between stocks:
print("\nCorrelation Matrix between stocks in 2023:")
print(returns_wide.corr())

Wide Returns DataFrame (first 5 rows):
Ticker          AAPL      AMZN     GOOGL      MSFT      NVDA
Date                                                        
2023-01-03       NaN       NaN       NaN       NaN       NaN
2023-01-04  0.010314 -0.007924 -0.011670 -0.043743  0.030318
2023-01-05 -0.010605 -0.023726 -0.021344 -0.029638 -0.032816
2023-01-06  0.036794  0.035611  0.013225  0.011785  0.041640
2023-01-09  0.004089  0.014870  0.007786  0.009736  0.051753

Correlation Matrix between stocks in 2023:
Ticker      AAPL      AMZN     GOOGL      MSFT      NVDA
Ticker                                                  
AAPL    1.000000  0.441677  0.528273  0.547987  0.444879
AMZN    0.441677  1.000000  0.600858  0.575928  0.380412
GOOGL   0.528273  0.600858  1.000000  0.509886  0.405933
MSFT    0.547987  0.575928  0.509886  1.000000  0.537431
NVDA    0.444879  0.380412  0.405933  0.537431  1.000000


> **Key Takeaway**: `.unstack(level="Ticker")` pivots long MultiIndex series into wide DataFrames, enabling cross-sectional comparisons and correlation analysis.

---

## Topic 6: Creating a Production Feature Function

To make our quantitative workflow reproducible, we encapsulate our entire feature generation process into a clean helper: `add_features()`.

Our production function generates:
- `Return_1d`: 1-day percentage return.
- `Return_20d`: 20-day cumulative momentum return.
- `Volatility_20d`: 20-day rolling return volatility.
- `SMA_50`: 50-day Simple Moving Average.

Let's test our complete feature engineering function. Let's see:

In [6]:
# Production-ready feature engineering function:
def add_features(df):
    df = df.copy()
    grouped = df.groupby(level="Ticker")
    df["Return_1d"] = grouped["Close"].pct_change()
    df["Return_20d"] = grouped["Close"].pct_change(20)
    df["Volatility_20d"] = grouped["Return_1d"].transform(
        lambda x: x.rolling(20).std()
    )
    df["SMA_50"] = grouped["Close"].transform(
        lambda x: x.rolling(50).mean()
    )
    return df.dropna()

# Test it on the universe:
featured_universe = add_features(universe)

# Print the shape before and after:
print(f"Shape before feature engineering: {universe.shape}")
print(f"Shape after feature engineering:  {featured_universe.shape}")
print("\nFirst 10 rows of featured universe:")
print(featured_universe[["Close", "Return_1d", "Return_20d", "Volatility_20d", "SMA_50"]].head(10))

Shape before feature engineering: (1250, 10)
Shape after feature engineering:  (1005, 14)

First 10 rows of featured universe:
Price                   Close  Return_1d  Return_20d  Volatility_20d      SMA_50
Date       Ticker                                                           
2023-03-15 AAPL    150.536621   0.002598   -0.007621        0.015159  142.358197
           AMZN     96.199997   0.013912   -0.035397        0.017820   95.929000
           GOOGL    95.265266  -0.003975    0.015886        0.017120   93.493374
           MSFT    258.252716   0.017811   -0.026417        0.015982  242.450012
           NVDA     24.178391  -0.008436    0.061730        0.040009   20.412386
2023-03-16 AAPL    153.350739   0.018694    0.000326        0.015425  142.967677
           AMZN    100.040001   0.039917    0.002204        0.019819   96.213400
           GOOGL    99.438255   0.043793    0.038148        0.019005   93.715405
           MSFT    268.721344   0.040536    0.003738        0.01837

> **Key Takeaway**: Encapsulating grouped feature pipelines into modular functions guarantees data integrity and clean code reusability.

---

## Practice Time

Now it is your turn to practice grouped operations on MultiIndex datasets. Guarding against cross-asset leakage is essential for production quants, so work through these challenges deliberately.

---

### Challenge 1: Calculating Grouped Multi-Asset Returns

- Build a 5-stock universe (`["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]`) across 2023.
- Add daily percentage returns using `.groupby(level="Ticker")`.
- Display the first 10 rows.

In [ ]:
# Challenge 1: Build universe of 5 stocks and add daily returns using .groupby(level="Ticker")
# Write your code below this line:




### Challenge 2: Grouped Rolling Volatility with .transform()

- Add a 10-day rolling volatility column named `"Volatility_10d"` using `.groupby(level="Ticker")` and `.transform()`.
- Display the first 10 populated rows.

In [ ]:
# Challenge 2: Add a 10-day rolling volatility column
# Write your code below this line:




### Challenge 3: Pivoting and Visualizing Normalized Prices

- Use `.unstack()` to create a wide DataFrame of closing prices named `prices_wide`.
- Normalize the prices to start at 1.0 on day 1 (`prices_wide / prices_wide.iloc[0]`).
- Plot all five normalized stocks on a single comparison chart.

In [ ]:
# Challenge 3: Use .unstack() to create wide closing prices DataFrame and plot all 5 stocks
# Write your code below this line:




---

## Solutions Section

Terrific work completing these multi-asset grouped challenges! Ensuring strict separation between assets during feature calculation keeps your backtesting honest.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
ex_universe = build_universe(tickers, "2023-01-01", "2024-01-01")
ex_universe["Return"] = ex_universe.groupby(level="Ticker")["Close"].pct_change()

print("Exercise 1 Result (first 10 rows):")
print(ex_universe[["Close", "Return"]].head(10))
```

#### Solution for Challenge 2:
```python
ex_universe["Volatility_10d"] = ex_universe.groupby(level="Ticker")["Return"].transform(
    lambda x: x.rolling(10).std()
)

print("Exercise 2 Result (first 10 populated rows):")
print(ex_universe[["Close", "Return", "Volatility_10d"]].dropna().head(10))
```

#### Solution for Challenge 3:
```python
prices_wide = ex_universe["Close"].unstack(level="Ticker")
normalized_prices = prices_wide / prices_wide.iloc[0]

plt.figure(figsize=(12, 6))
plt.plot(normalized_prices)
plt.legend(normalized_prices.columns)
plt.title("2023 Normalized Stock Performance (Base = 1.0)")
plt.ylabel("Growth Multiple")
plt.show()
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [7]:
# Solution for Challenge 1:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
ex_universe = build_universe(tickers, "2023-01-01", "2024-01-01")
ex_universe["Return"] = ex_universe.groupby(level="Ticker")["Close"].pct_change()

print("Exercise 1 Result (first 10 rows):")
print(ex_universe[["Close", "Return"]].head(10))

Exercise 1 Result (first 10 rows):
Price                   Close    Return
Date       Ticker                      
2023-01-03 AAPL    122.876732       NaN
           AMZN     85.820000       NaN
           GOOGL    88.336708       NaN
           MSFT    232.510559       NaN
           NVDA     14.283262       NaN
2023-01-04 AAPL    124.144127  0.010314
           AMZN     85.139999 -0.007924
           GOOGL    87.305840 -0.011670
           MSFT    222.339813 -0.043743
           NVDA     14.716300  0.030318


In [8]:
# Solution for Challenge 2:
ex_universe["Volatility_10d"] = ex_universe.groupby(level="Ticker")["Return"].transform(
    lambda x: x.rolling(10).std()
)

print("Exercise 2 Result (first 10 populated rows):")
print(ex_universe[["Close", "Return", "Volatility_10d"]].dropna().head(10))

Exercise 2 Result (first 10 populated rows):
Price                   Close    Return  Volatility_10d
Date       Ticker                                      
2023-01-18 AAPL    132.868774 -0.005370        0.016335
           AMZN     95.459999 -0.006039        0.030574
           GOOGL    91.319763 -0.001962        0.018804
           MSFT    231.272186 -0.018895        0.024479
           NVDA     17.348633 -0.018408        0.032607
2023-01-19 AAPL    132.798889 -0.000526        0.016206
           AMZN     93.680000 -0.018647        0.030799
           GOOGL    92.810577  0.016325        0.018970
           MSFT    227.464951 -0.016466        0.020583
           NVDA     16.744158 -0.034843        0.034636


In [9]:
# Solution for Challenge 3:
prices_wide = ex_universe["Close"].unstack(level="Ticker")
normalized_prices = prices_wide / prices_wide.iloc[0]

print("Displaying normalized prices chart for all 5 stocks:")
plt.figure(figsize=(12, 6))
plt.plot(normalized_prices)
plt.legend(normalized_prices.columns)
plt.title("2023 Normalized Stock Performance (Base = 1.0)")
plt.ylabel("Growth Multiple")
plt.show()

Displaying normalized prices chart for all 5 stocks:
